# 5-Step DuPont Decomposition — Greek Systemic Banks (2022–2024)

Banking version of the DuPont identity. Corporate DuPont starts from EBIT, which has no meaning for a bank, so the chain starts from pre-provision operating profit (PPOP):

$$\text{ROE} = \underbrace{\frac{\text{NP}}{\text{PBT}}}_{\text{Tax and other}} \times \underbrace{\frac{\text{PBT}}{\text{PPOP}}}_{\text{Provision burden}} \times \underbrace{\frac{\text{PPOP}}{\text{Op. income}}}_{\text{Pre-provision margin}} \times \underbrace{\frac{\text{Op. income}}{\text{Avg assets}}}_{\text{Asset yield}} \times \underbrace{\frac{\text{Avg assets}}{\text{Avg equity}}}_{\text{Leverage}}$$

- **PPOP** = operating income − operating expenses, as printed in each bank's income statement.
- **NP** = net profit attributable to shareholders (includes discontinued operations), so the first lever also captures discontinued operations and minorities, not only tax.
- Assets and equity are averages of opening and closing balances, the same basis as the ROE in `kpis_final.csv`, so the five levers multiply back to ROE exactly.

**Data:** consolidated statements of the four listed groups (Alpha Services and Holdings, Eurobank Holdings, NBG, Piraeus Financial Holdings), every figure traced to a PDF page in `02_Banking_Sector_Dashboard/data/processed/sources.csv`.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

# ── Brand colors (bank logo palette) ──────────────────────────────────────────
COLORS = {
    'Eurobank':    '#0067B1',
    'Alpha Bank':  '#E2231A',
    'Piraeus Bank':'#F7A600',
    'NBG':         '#003087',
}
BANKS  = list(COLORS.keys())
YEARS  = [2022, 2023, 2024]

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR = Path('../02_Banking_Sector_Dashboard/data/processed')

print('Libraries loaded. Data directory:', DATA_DIR.resolve())

Libraries loaded. Data directory: /home/claude/greek-banking-sector-analysis/02_Banking_Sector_Dashboard/data/processed


In [2]:
# ── Load processed data ───────────────────────────────────────────────────────
kpis = pd.read_csv(DATA_DIR / 'kpis_final.csv')
is_long = pd.read_csv(DATA_DIR / 'income_statement_final.csv')

is_pivot = (
    is_long
    .pivot_table(index=['bank', 'year'], columns='metric', values='value', aggfunc='first')
    .reset_index()
)
df = kpis.merge(
    is_pivot[['bank', 'year', 'Profit from operations before impairments']]
        .rename(columns={'Profit from operations before impairments': 'ppop'}),
    on=['bank', 'year'], how='left'
).rename(columns={'pbt': 'profit_before_tax'})

print(f'Loaded {len(df)} bank-year observations.')
df[['bank', 'year', 'net_profit', 'profit_before_tax', 'ppop', 'operating_income',
    'avg_total_assets', 'avg_equity', 'roe']].round(1)

Loaded 12 bank-year observations.


,bank,year,net_profit,profit_before_tax,ppop,operating_income,avg_total_assets,avg_equity,roe
0,Alpha Bank,2022,368.1,543.9,894.1,1750.8,75683.7,6140.4,6.0
1,Alpha Bank,2023,618.0,780.0,1314.0,2129.0,75216.2,6775.0,9.1
2,Alpha Bank,2024,654.0,816.0,1356.0,2222.0,72248.0,7739.0,8.5
3,Eurobank,2022,1347.0,1751.0,2184.0,3041.0,79663.0,6088.0,22.1
4,Eurobank,2023,1140.0,1542.0,1999.0,2914.0,80627.5,7268.0,15.7
5,Eurobank,2024,1448.0,1872.0,2242.0,3341.0,90465.5,8399.0,17.2
6,NBG,2022,1120.0,1155.0,1500.0,2355.0,81035.5,6101.0,18.4
7,NBG,2023,1106.0,1479.0,1832.0,2760.0,76348.5,7039.0,15.7
8,NBG,2024,1158.0,1517.0,1832.0,2885.0,74770.5,8024.5,14.4
9,Piraeus Bank,2022,950.0,1037.0,1693.0,2582.0,77217.0,6170.5,15.4


In [3]:
# ── Compute 5-step DuPont components ─────────────────────────────────────────
df['tax_burden']          = df['net_profit']        / df['profit_before_tax']  # (1) NP / PBT
df['provision_burden']    = df['profit_before_tax'] / df['ppop']               # (2) PBT / PPOP
df['preprovision_margin'] = df['ppop']              / df['operating_income']   # (3) PPOP / Op. income
df['asset_utilisation']   = df['operating_income']  / df['avg_total_assets']   # (4) Op. income / avg assets
df['leverage']            = df['avg_total_assets']  / df['avg_equity']         # (5) avg assets / avg equity

df['roe_reconstructed'] = (
    df['tax_burden'] * df['provision_burden'] * df['preprovision_margin']
    * df['asset_utilisation'] * df['leverage'] * 100
)

components = ['tax_burden', 'provision_burden', 'preprovision_margin', 'asset_utilisation', 'leverage']
print('DuPont components computed.')
df[['bank', 'year'] + components + ['roe_reconstructed', 'roe']].round(4)

DuPont components computed.


,bank,year,tax_burden,provision_burden,preprovision_margin,asset_utilisation,leverage,roe_reconstructed,roe
0,Alpha Bank,2022,0.6768,0.6083,0.5107,0.0231,12.3256,5.9948,6.0
1,Alpha Bank,2023,0.7923,0.5936,0.6172,0.0283,11.1021,9.1218,9.1
2,Alpha Bank,2024,0.8015,0.6018,0.6103,0.0308,9.3356,8.4507,8.5
3,Eurobank,2022,0.7693,0.8017,0.7182,0.0382,13.0852,22.1255,22.1
4,Eurobank,2023,0.7393,0.7714,0.6860,0.0361,11.0935,15.6852,15.7
5,Eurobank,2024,0.7735,0.8350,0.6711,0.0369,10.7710,17.2401,17.2
6,NBG,2022,0.9697,0.7700,0.6369,0.0291,13.2823,18.3576,18.4
7,NBG,2023,0.7478,0.8073,0.6638,0.0362,10.8465,15.7125,15.7
8,NBG,2024,0.7633,0.8281,0.6350,0.0386,9.3178,14.4308,14.4
9,Piraeus Bank,2022,0.9161,0.6125,0.6557,0.0334,12.5139,15.3958,15.4


In [4]:
# ── Check: the five levers must multiply back to the ROE in kpis_final.csv ────
df['roe_error_pp'] = (df['roe_reconstructed'] - df['roe']).abs()

MAX_TOLERANCE_PP = 0.06   # ROE is stored with one decimal
failures = df[df['roe_error_pp'] > MAX_TOLERANCE_PP][['bank', 'year', 'roe', 'roe_reconstructed', 'roe_error_pp']]

print(f'ROE reconstruction check (tolerance ±{MAX_TOLERANCE_PP}pp)')
print(df[['bank', 'year', 'roe', 'roe_reconstructed', 'roe_error_pp']].round(3).to_string(index=False))

if len(failures) > 0:
    raise AssertionError(f'{len(failures)} bank-year(s) do not reconcile. Investigate data.')
print(f'\nAll 12 bank-years reconcile within ±{MAX_TOLERANCE_PP}pp.')

ROE reconstruction check (tolerance ±0.06pp)
        bank  year  roe  roe_reconstructed  roe_error_pp
  Alpha Bank  2022  6.0              5.995         0.005
  Alpha Bank  2023  9.1              9.122         0.022
  Alpha Bank  2024  8.5              8.451         0.049
    Eurobank  2022 22.1             22.125         0.025
    Eurobank  2023 15.7             15.685         0.015
    Eurobank  2024 17.2             17.240         0.040
         NBG  2022 18.4             18.358         0.042
         NBG  2023 15.7             15.712         0.012
         NBG  2024 14.4             14.431         0.031
Piraeus Bank  2022 15.4             15.396         0.004
Piraeus Bank  2023 11.4             11.378         0.022
Piraeus Bank  2024 13.7             13.742         0.042

All 12 bank-years reconcile within ±0.06pp.


In [5]:
# ── Chart 1: Small Multiples — Component Evolution Per Bank ──────────────────
# For each bank: how did each DuPont lever change from 2022 → 2024?
# Normalised to 2022 = 100 so all 5 levers are comparable on one axis.

COMP_LABELS = {
    'tax_burden':          'Tax Burden (NP/PBT)',
    'provision_burden':    'Provision Burden (PBT/PPOP)',
    'preprovision_margin': 'Pre-provision Margin (PPOP/OpInc)',
    'asset_utilisation':   'Asset Utilisation (OpInc/Assets)',
    'leverage':            'Leverage (Assets/Equity)',
}
COMP_COLORS = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA', '#FFA15A']

fig1 = make_subplots(
    rows=2, cols=2,
    subplot_titles=[f'<b>{b}</b>' for b in BANKS],
    shared_xaxes=False,
    vertical_spacing=0.14,
    horizontal_spacing=0.10,
)

for bi, bank in enumerate(BANKS):
    row, col = divmod(bi, 2)
    bank_df = df[df['bank'] == bank].sort_values('year')
    base = bank_df.set_index('year')[components].loc[2022]  # 2022 base

    for ci, (comp, color) in enumerate(zip(components, COMP_COLORS)):
        indexed = (bank_df[comp].values / base[comp]) * 100
        fig1.add_trace(
            go.Scatter(
                x=YEARS, y=indexed,
                name=COMP_LABELS[comp],
                line=dict(color=color, width=2),
                marker=dict(size=7),
                showlegend=(bi == 0),
                legendgroup=comp,
            ),
            row=row + 1, col=col + 1,
        )

    # Overlay reconstructed ROE trend (dashed)
    roe_indexed = (bank_df['roe_reconstructed'].values / bank_df['roe_reconstructed'].values[0]) * 100
    fig1.add_trace(
        go.Scatter(
            x=YEARS, y=roe_indexed,
            name='ROE (indexed)',
            line=dict(color='white', width=2.5, dash='dash'),
            marker=dict(size=8, symbol='diamond'),
            showlegend=(bi == 0),
            legendgroup='roe',
        ),
        row=row + 1, col=col + 1,
    )

    # Add 100 baseline
    fig1.add_hline(y=100, line_dash='dot', line_color='rgba(255,255,255,0.3)', row=row + 1, col=col + 1)

fig1.update_layout(
    title_text='<b>DuPont Component Evolution</b> | Indexed to 2022 = 100 | Greek Systemic Banks',
    title_font_size=16,
    template='plotly_dark',
    height=620,
    legend=dict(orientation='h', yanchor='bottom', y=-0.22, x=0.5, xanchor='center', font_size=11),
    paper_bgcolor='#0f1117',
    plot_bgcolor='#0f1117',
)
fig1.update_xaxes(tickvals=YEARS, ticktext=[str(y) for y in YEARS])
fig1.update_yaxes(title_text='Index (2022=100)', ticksuffix='')
fig1.show()

In [6]:
# ── Chart 2: Cross-Bank Comparison Per DuPont Lever (2024) ───────────────────
# For 2024: which bank leads on each lever? This is the equity-research comparison.

df24 = df[df['year'] == 2024].set_index('bank')

# Use 2×3 layout (5 components + ROE)
fig2 = make_subplots(
    rows=2, cols=3,
    subplot_titles=[
        'Tax Burden (NP/PBT)',
        'Provision Burden (PBT/PPOP)',
        'Pre-provision Margin (PPOP/OpInc)',
        'Asset Yield (OpInc/Avg Assets, %)',
        'Leverage (Avg Assets/Avg Equity, ×)',
        'ROE (%)',
    ],
    vertical_spacing=0.18,
    horizontal_spacing=0.10,
)

metrics = components + ['roe']
positions = [(1,1),(1,2),(1,3),(2,1),(2,2),(2,3)]
multipliers = [1, 1, 1, 100, 1, 1]  # asset utilisation shown as %

for (metric, (r, c), mult) in zip(metrics, positions, multipliers):
    values = [df24.loc[b, metric] * mult for b in BANKS if b in df24.index]
    bar_colors = [COLORS[b] for b in BANKS if b in df24.index]
    fig2.add_trace(
        go.Bar(
            x=BANKS,
            y=values,
            marker_color=bar_colors,
            text=[f'{v:.2f}' if mult == 1 else f'{v:.2f}%' for v in values],
            textposition='outside',
            showlegend=False,
        ),
        row=r, col=c,
    )

fig2.update_layout(
    title_text='<b>DuPont Lever Comparison — 2024</b> | Greek Systemic Banks',
    title_font_size=16,
    template='plotly_dark',
    height=600,
    paper_bgcolor='#0f1117',
    plot_bgcolor='#0f1117',
)
fig2.update_xaxes(tickfont_size=11)
fig2.show()

In [7]:
# ── Chart 3: ROE Bridge — What drove ROE differences in 2024? ────────────────
# Compare each bank vs sector median on each DuPont lever.
# Shows which bank is above / below peer median and by how much (in log-contribution).

df24_comp = df[df['year'] == 2024].copy()
sector_medians = df24_comp[components].median()

# Log-contribution relative to sector median (sum = ROE premium over median)
for comp in components:
    df24_comp[f'log_{comp}'] = np.log(df24_comp[comp] / sector_medians[comp])

log_cols = [f'log_{c}' for c in components]
df24_comp['total_log_premium'] = df24_comp[log_cols].sum(axis=1)

fig3 = go.Figure()
bar_width = 0.15
for bi, bank in enumerate(BANKS):
    row_b = df24_comp[df24_comp['bank'] == bank].iloc[0]
    log_vals = [row_b[f'log_{c}'] for c in components]
    x_pos = np.arange(len(components)) + bi * bar_width - (len(BANKS) - 1) * bar_width / 2
    fig3.add_trace(go.Bar(
        x=x_pos,
        y=log_vals,
        name=bank,
        marker_color=COLORS[bank],
        width=bar_width,
        text=[f'{v:+.3f}' for v in log_vals],
        textposition='outside',
        textfont_size=9,
    ))

fig3.add_hline(y=0, line_color='rgba(255,255,255,0.4)', line_dash='dash')
comp_abbrev = ['Tax Burden', 'Provision Burden', 'Pre-prov. Margin', 'Asset Util.', 'Leverage']
fig3.update_layout(
    title_text='<b>ROE Driver Analysis — 2024</b> | Log premium/discount vs sector median per DuPont lever',
    title_font_size=15,
    template='plotly_dark',
    height=430,
    xaxis=dict(
        tickmode='array',
        tickvals=np.arange(len(components)),
        ticktext=comp_abbrev,
    ),
    yaxis_title='Log premium vs sector median (+ = above median)',
    legend=dict(orientation='h', y=-0.20, x=0.5, xanchor='center'),
    barmode='group',
    paper_bgcolor='#0f1117',
    plot_bgcolor='#0f1117',
)
fig3.show()

In [8]:
# ── Key Insights Table ────────────────────────────────────────────────────────
print('=' * 72)
print('  DuPont Summary — Greek Systemic Banks (2022–2024)')
print('=' * 72)

display_cols = [
    ('bank', 'Bank'), ('year', 'Year'),
    ('tax_burden', 'Tax Burden'), ('provision_burden', 'Prov. Burden'),
    ('preprovision_margin', 'PreProv. Margin'), ('asset_utilisation', 'Asset Util. (%)'),
    ('leverage', 'Leverage (×)'), ('roe', 'ROE (%)'),
]

out = df[['bank','year','tax_burden','provision_burden','preprovision_margin','asset_utilisation','leverage','roe']].copy()
out['asset_utilisation'] = (out['asset_utilisation'] * 100).round(2)
out[['tax_burden','provision_burden','preprovision_margin']] = out[['tax_burden','provision_burden','preprovision_margin']].round(3)
out['leverage'] = out['leverage'].round(2)
out['roe'] = out['roe'].round(1)

print(out.to_string(index=False))

print('\n── Key Findings ──')

# Find who has the highest/lowest of each component in 2024
for comp, label in zip(components, comp_abbrev):
    mult = 100 if comp == 'asset_utilisation' else 1
    best_bank = df24_comp.set_index('bank')[comp].idxmax()
    best_val  = df24_comp.set_index('bank')[comp].max() * mult
    worst_bank = df24_comp.set_index('bank')[comp].idxmin()
    worst_val  = df24_comp.set_index('bank')[comp].min() * mult
    unit = '%' if comp == 'asset_utilisation' else '×' if comp == 'leverage' else ''
    fmt = '.2f' if comp in ('asset_utilisation', 'leverage') else '.3f'
    print(f'  {label:<22}  Best: {best_bank} ({best_val:{fmt}}{unit})   '
          f'Worst: {worst_bank} ({worst_val:{fmt}}{unit})')

  DuPont Summary — Greek Systemic Banks (2022–2024)
        bank  year  tax_burden  provision_burden  preprovision_margin  asset_utilisation  leverage  roe
  Alpha Bank  2022       0.677             0.608                0.511               2.31     12.33  6.0
  Alpha Bank  2023       0.792             0.594                0.617               2.83     11.10  9.1
  Alpha Bank  2024       0.801             0.602                0.610               3.08      9.34  8.5
    Eurobank  2022       0.769             0.802                0.718               3.82     13.09 22.1
    Eurobank  2023       0.739             0.771                0.686               3.61     11.09 15.7
    Eurobank  2024       0.774             0.835                0.671               3.69     10.77 17.2
         NBG  2022       0.970             0.770                0.637               2.91     13.28 18.4
         NBG  2023       0.748             0.807                0.664               3.62     10.85 15.7
         NBG

In [9]:
# ── Final validation ──────────────────────────────────────────────────────────
assert len(df) == 12, 'Expected 12 bank-year rows (4 banks × 3 years).'
assert df['profit_before_tax'].notna().all(), 'Missing profit before tax.'
assert df['roe_error_pp'].max() <= 0.06, 'ROE reconstruction error above rounding tolerance.'
for comp in components:
    assert (df[comp] > 0).all(), f'Non-positive value in component: {comp}'

print('All checks passed.')
print(f'   Max ROE reconstruction error: {df["roe_error_pp"].max():.4f} pp (rounding only)')

All checks passed.
   Max ROE reconstruction error: 0.0493 pp (rounding only)
